# Dual Spam Model Training (Definitive .npy Exporter)

**FINAL, GUARANTEED SOLUTION:** This notebook's ONLY purpose is to train the models and export the raw weights for each layer as individual `.npy` files. This bypasses all buggy TensorFlow.js converters and automatic loaders. It will produce a `models.zip` file for download.

### Step 1: Install Dependencies and Upload Datasets

In [ ]:
!pip install -q tensorflow==2.15.0 pandas scikit-learn numpy

from google.colab import files
import tensorflow as tf
import json
import os
import numpy as np
import shutil

print(f"TensorFlow Version: {tf.__version__}")

print("\nPlease upload your two datasets: spam.csv and spaml.csv")
uploaded = files.upload()

if 'spam.csv' in uploaded and 'spaml.csv' in uploaded:
    print("\n✅ Successfully uploaded both files!")
else:
    print("\n❌ Error: Please make sure you upload both files.")

---
# Part 1: SMS Model Training & .npy Export

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import io
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Create a directory for the final model files
output_dir = 'models'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

# 1. Load & Preprocess Data
df_sms = pd.read_csv(io.BytesIO(uploaded['spam.csv']), encoding='latin1')
df_sms = df_sms[['v1', 'v2']]
df_sms.columns = ['label', 'text']
df_sms['label'] = df_sms['label'].map({'ham': 0, 'spam': 1})
texts = df_sms['text'].values
labels_sms = df_sms['label'].values
vocab_size_sms = 10000
max_length_sms = 150
embedding_dim_sms = 16
tokenizer_sms = Tokenizer(num_words=vocab_size_sms, oov_token='<OOV>')
tokenizer_sms.fit_on_texts(texts)
padded_sms = pad_sequences(tokenizer_sms.texts_to_sequences(texts), maxlen=max_length_sms, padding='post', truncating='post')

# 2. Build & Train Model
X_train_sms, X_test_sms, y_train_sms, y_test_sms = train_test_split(padded_sms, labels_sms, test_size=0.2, random_state=42)
input_sms = tf.keras.Input(shape=(max_length_sms,))
x = tf.keras.layers.Embedding(name='embedding_sms', input_dim=vocab_size_sms, output_dim=embedding_dim_sms)(input_sms)
x = tf.keras.layers.GlobalAveragePooling1D(name='pooling_sms')(x)
x = tf.keras.layers.Dense(24, activation='relu', name='dense_1_sms')(x)
x = tf.keras.layers.Dropout(0.2, name='dropout_sms')(x)
output_sms = tf.keras.layers.Dense(1, activation='sigmoid', name='output_sms')(x)
model_sms = tf.keras.Model(inputs=input_sms, outputs=output_sms)
model_sms.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_sms.fit(X_train_sms, y_train_sms, epochs=10, validation_data=(X_test_sms, y_test_sms), verbose=0)
print("✅ SMS Model Trained.")

# 3. ✨ Manually Save Each Layer's Weights as .npy ✨
for layer in model_sms.layers:
    weights = layer.get_weights()
    if weights:
        for i, w in enumerate(weights):
            filename = f"{layer.name}_{i}.npy"
            np.save(os.path.join(output_dir, filename), w)
            print(f"Saved {filename}")

# 4. Save tokenizer index
with open(os.path.join(output_dir, 'sms_word_index.json'), 'w') as f:
    json.dump(tokenizer_sms.word_index, f)
print("Saved sms_word_index.json")

---
# Part 2: URL Model Training & .npy Export

In [ ]:
# 1. Load & Preprocess Data
df_url = pd.read_csv(io.BytesIO(uploaded['spaml.csv']))
df_url['is_spam'] = df_url['is_spam'].astype(int)
urls = df_url['url'].values
labels_url = df_url['is_spam'].values
vocab_size_url = 128
max_length_url = 200
embedding_dim_url = 16
tokenizer_url = Tokenizer(num_words=vocab_size_url, char_level=True, oov_token='<OOV>')
tokenizer_url.fit_on_texts(urls)
padded_url = pad_sequences(tokenizer_url.texts_to_sequences(urls), maxlen=max_length_url, padding='post', truncating='post')

# 2. Build & Train Model
X_train_url, X_test_url, y_train_url, y_test_url = train_test_split(padded_url, labels_url, test_size=0.2, random_state=42)
input_url = tf.keras.Input(shape=(max_length_url,))
y = tf.keras.layers.Embedding(name='embedding_url', input_dim=vocab_size_url, output_dim=embedding_dim_url)(input_url)
y = tf.keras.layers.GlobalAveragePooling1D(name='pooling_url')(y)
y = tf.keras.layers.Dense(24, activation='relu', name='dense_1_url')(y)
y = tf.keras.layers.Dropout(0.2, name='dropout_url')(y)
output_url = tf.keras.layers.Dense(1, activation='sigmoid', name='output_url')(y)
model_url = tf.keras.Model(inputs=input_url, outputs=output_url)
model_url.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_url.fit(X_train_url, y_train_url, epochs=20, validation_data=(X_test_url, y_test_url), verbose=0)
print("✅ URL Model Trained.")

# 3. ✨ Manually Save Each Layer's Weights as .npy ✨
for layer in model_url.layers:
    weights = layer.get_weights()
    if weights:
        for i, w in enumerate(weights):
            filename = f"{layer.name}_{i}.npy"
            np.save(os.path.join(output_dir, filename), w)
            print(f"Saved {filename}")

# 4. Save tokenizer index
with open(os.path.join(output_dir, 'url_char_index.json'), 'w') as f:
    json.dump(tokenizer_url.word_index, f)
print("Saved url_char_index.json")

# 5. Zip the entire models folder for download
shutil.make_archive('models', 'zip', output_dir)
print("\n✅ All files saved and zipped.")
print("Downloading models.zip...")
files.download('models.zip')